In [1]:
import pandas as pd

# 1. Descargamos los datos reales del Titanic desde internet
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
datos_titanic = pd.read_csv(url)

# 2. Le pedimos a Pandas que nos muestre las primeras 5 filas de la tabla
datos_titanic.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
# Contamos cuántos datos faltan (valores nulos) en cada columna
datos_titanic.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Embarked         2
dtype: int64

In [4]:
# .drop elimina lo que le pidamos. 
# axis=1 le dice a Pandas que 'Cabin' es una columna (vertical), no una fila (horizontal).
datos_titanic = datos_titanic.drop('Cabin', axis=1)

KeyError: "['Cabin'] not found in axis"

In [6]:
# 1. Calculamos la edad promedio de los pasajeros que sí tienen este dato
edad_promedio = datos_titanic['Age'].mean()

# 2. Rellenamos los huecos vacíos de la columna Age con ese promedio
datos_titanic['Age'] = datos_titanic['Age'].fillna(edad_promedio)

In [7]:
# Usamos .map() para buscar palabras específicas y reemplazarlas por números
datos_titanic['Sex'] = datos_titanic['Sex'].map({'male': 0, 'female': 1})

# Damos un vistazo rápido a las primeras filas para confirmar los cambios
datos_titanic.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,S


In [8]:
# 1. Para X: Eliminamos la respuesta ('Survived') y las columnas de "ruido"
X = datos_titanic.drop(['Survived', 'PassengerId', 'Name', 'Ticket'], axis=1)

# 2. Para y: Seleccionamos ÚNICAMENTE la columna de la respuesta
y = datos_titanic['Survived']

# Vemos cómo quedó nuestra tabla final de pistas (X)
X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,0,22.0,1,0,7.2500,S
1,1,1,38.0,1,0,71.2833,C
2,3,1,26.0,0,0,7.9250,S
3,1,1,35.0,1,0,53.1000,S
4,3,0,35.0,0,0,8.0500,S


In [9]:
from sklearn.model_selection import train_test_split

# Separamos los datos. test_size=0.2 significa que el 20% será para el examen.
# random_state=42 asegura que el reparto al azar sea el mismo en tu PC y en la mía.
X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(X, y, test_size=0.2, random_state=42)

print("Pasajeros para estudiar (80%):", len(X_entrenamiento))
print("Pasajeros para el examen (20%):", len(X_prueba))

Pasajeros para estudiar (80%): 712
Pasajeros para el examen (20%): 179


In [11]:
X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,0,22.0,1,0,7.2500,S
1,1,1,38.0,1,0,71.2833,C
2,3,1,26.0,0,0,7.9250,S
3,1,1,35.0,1,0,53.1000,S
4,3,0,35.0,0,0,8.0500,S


In [10]:
from sklearn.tree import DecisionTreeClassifier

# 1. Creamos el modelo (un árbol de decisión vacío)
# max_depth=3 limita el árbol a un máximo de 3 niveles de preguntas para evitar el sobreajuste
modelo_titanic = DecisionTreeClassifier(max_depth=3, random_state=42)

# 2. Entrenamos el modelo pasándole las pistas y las respuestas de estudio
modelo_titanic.fit(X_entrenamiento, y_entrenamiento)

ValueError: could not convert string to float: 'S'

In [12]:
# 1. Transformamos la columna 'Embarked' en columnas de ceros y unos
X = pd.get_dummies(X, columns=['Embarked'])

# 2. Volvemos a dividir los datos (porque nuestra X acaba de cambiar)
X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. ¡Entrenamos el modelo de nuevo! (Esta vez sin letras que lo confundan)
modelo_titanic.fit(X_entrenamiento, y_entrenamiento)

DecisionTreeClassifier(max_depth=3, random_state=42)

In [13]:
# Pedimos al modelo que adivine el destino de los pasajeros ocultos
predicciones = modelo_titanic.predict(X_prueba)

# Mostramos las primeras 10 respuestas de su examen (0 = No sobrevivió, 1 = Sobrevivió)
print(predicciones[:10])

[0 0 0 1 1 1 1 0 1 1]


In [14]:
from sklearn.metrics import accuracy_score

# Comparamos las respuestas reales (y_prueba) contra las adivinanzas del modelo (predicciones)
calificacion = accuracy_score(y_prueba, predicciones)

# Multiplicamos por 100 para leerlo fácilmente como un porcentaje
print(f"Precisión del modelo: {calificacion * 100:.2f}%")

Precisión del modelo: 79.89%


In [15]:
# Creamos una pequeña tabla para ver qué pistas tuvieron más peso en el examen
importancias = pd.DataFrame({
    'Pista': X_entrenamiento.columns,
    'Importancia': modelo_titanic.feature_importances_
})

# Ordenamos la tabla de mayor a menor importancia
importancias.sort_values(by='Importancia', ascending=False).head()

,Pista,Importancia
1,Sex,0.605737
0,Pclass,0.209536
2,Age,0.075353
5,Fare,0.061240
3,SibSp,0.048135
